In [1]:
#try to use brain and write some code to use controller in sharpy 
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import sharpy.cases.templates.flying_wings as wings
import sharpy.sharpy_main
import sharpy.utils.plotutils as pu

u_inf = 25.
alpha_deg = 2.  # Define angle of attack for static aeroelastic analysis.
rho = 1.225      # Air density. #inquire about this for our wind tunnel

M = 4             # Number of chordwise panels
N = 16            # Number of spanwise panels
M_star_fact = 10   # Length of the wake in chords.

case_name = 'mycaseControlThree'
route_test_dir = os.path.abspath('')

print('The case to run will be: %s' % case_name)
print('Case files will be saved in ./cases/%s' %case_name)
print('Output files will be saved in ./output/%s/' %case_name)

#let us assume we are making a wing that is half the size, thus let us set our b_ref to 1.2 metres
#to achieve an aspect ratio of 12 we need 2*1.2/x = 12 leading to our chord of 0.2 metres

#the elastic axis is at 0.33 chord and the main cg is at 0.43 chord as per the goland wing
ws = wings.GolandControlSurface(M=M,
                                N=N,
                                Mstar_fact=M_star_fact,
                                u_inf=u_inf,
                                alpha=alpha_deg,
                                cs_deflection=[0,0],
                                n_control_surfaces=2,
                                rho=rho,
                                b_ref=2. * 1.2,
                                main_chord= 0.2,
                                pct_flap=0.2375, #285mm/1200mm
                                aspect_ratio=(2. * 1.2) / 0.2,
                                sweep=0,
                                cs_type=0, #flap?
                                n_surfaces=2,
                                physical_time=1.0,
                                route = route_test_dir + '/cases',
                                case_name=case_name
                                )

def new_update_mass_stiff(self):
    import sharpy.utils.algebra as algebra

    #seeing our wing has roughly 2% of the area of the goland wing we have to scale the 
    #structural properties appropriately to achieve flutter, (as lift forces are proportional
    #to the wing area)
    width = 100E-3
    height = 2E-3
    E = 69e9  # Young's modulus in Pascals (Pa)
    G = 26e9  # Shear modulus in Pascals (Pa)
    rho_2 = 2700

    # Cross-sectional properties
    A = width * height  # Area in m²
    Iy = (width * height**3) / 12  # Second moment of area about the y-axis (m⁴)
    Iz = (height * width**3) / 12  # Second moment of area about the z-axis (m⁴)
    print(Iy)
    print(Iz)
    J = 2*height**3*width/3  # Torsion constant approximation for rectangular section

    # Rigidity calculations
    EA = E * A
    GA = G * A
    GJ = G * J
    EIy = E * Iy
    EIz = E * Iz

    ea, ga = EA, GA
    gj = GJ
    eiy = EIy
    eiz = EIz
    base_stiffness = np.diag([ea, ga, ga, gj, eiy, eiz])
    self.stiffness = np.zeros((1, 6, 6))
    self.stiffness[0] = base_stiffness
    print(base_stiffness)
    
    m_unit = 1.42
    j_tors = (Iy+Iz)*rho_2
    print(Iy+Iz)
    print(j_tors)
    pos_cg_b = np.array([0., self.c_ref * (self.main_cg - self.main_ea), 0.])
    m_chi_cg = algebra.skew(m_unit * pos_cg_b)
    self.mass = np.zeros((1, 6, 6))
    self.mass[0, :, :] = np.diag([m_unit, m_unit, m_unit,
                                    j_tors, .1 * j_tors, .9 * j_tors])

    self.mass[0, :3, 3:] = m_chi_cg
    self.mass[0, 3:, :3] = -m_chi_cg

    self.elem_stiffness = np.zeros((self.num_elem_tot,), dtype=int)
    self.elem_mass = np.zeros((self.num_elem_tot,), dtype=int)

    self.gains = -np.array([1, 1, 1])
    main_chord = 0.2
    self.dt = main_chord/M/u_inf*1
    self.num_steps = int(30./self.dt)

ws.update_mass_stiff = new_update_mass_stiff.__get__(ws, wings.Goland)   

ws.clean_test_files()
ws.update_derived_params()
ws.set_default_config_dict()

ws.generate_aero_file()
ws.generate_fem_file()

The case to run will be: mycaseControlThree
Case files will be saved in ./cases/mycaseControlThree
Output files will be saved in ./output/mycaseControlThree/
6.666666666666668e-11
1.666666666666667e-07
[[1.38000000e+07 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 5.20000000e+06 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 5.20000000e+06 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 1.38666667e+01 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 4.60000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 1.15000000e+04]]
1.6673333333333336e-07
0.00045018000000000006


In [ ]:
ws.config['SHARPy'] = {
    'flow':
        ['BeamLoader', 'AerogridLoader',
         'StaticCoupled',
         'AerogridPlot',
         'BeamPlot',
         'DynamicCoupled'
         ],
    'case': ws.case_name, 'route': ws.route,
    'write_screen': 'on', 'write_log': 'on',    # Change to 'on' as neded.
    'log_folder': route_test_dir + '/output/',
    'log_file': ws.case_name + '.log'}

ws.config['BeamLoader'] = {
    'unsteady': 'off',
    'orientation': ws.quat}

ws.config['AerogridLoader'] = {
    'unsteady': 'on',
    'aligned_grid': 'on',
    'mstar': 1,
    'wake_shape_generator': 'StraightWake',
    'wake_shape_generator_input': {'u_inf': ws.u_inf,
                                    'u_inf_direction': ws.u_inf_direction,
                                    'dt': ws.dt}}

ws.config['StaticCoupled'] = {
    'print_info': 'on',
    'max_iter': 200,
    'n_load_steps': 5,
    'tolerance': 1e-5,
    'relaxation_factor': 0.1,
    'aero_solver': 'StaticUvlm',
    'aero_solver_settings': {
        'rho': ws.rho,
        'print_info': 'off',
        'horseshoe': 'on',
        'num_cores': 4,
        'n_rollup': 0,
        'velocity_field_generator': 'SteadyVelocityField',
        'velocity_field_input': {
            'u_inf': ws.u_inf,
            'u_inf_direction': ws.u_inf_direction},
        'vortex_radius' : 1e9},
    'structural_solver': 'NonLinearStatic',
    'structural_solver_settings': {'print_info': 'off',
                                   'max_iterations': 150,
                                   'num_load_steps': 5,
                                   'delta_curved': 1e-6,
                                   'min_delta': 1e-8,
                                   'gravity_on': 'on',
                                   'gravity': 9.81}}

ws.config['AerogridPlot'] = {'include_rbm': 'off',
                             'include_applied_forces': 'on',
                             'minus_m_star': 0}

ws.config['BeamPlot'] = {'include_rbm': 'off',
                         'include_applied_forces': 'on'}

ws.config['WriteVariablesTime'] = {'structure_variables': ['pos'],
                                        'structure_nodes': list(range(0, ws.num_node_surf)),
                                        'cleanup_old_solution': 'on',
                                        }

ws.config['DynamicCoupled'] = {'print_info': 'on',
                                  'structural_substeps': 10,
                                  'dynamic_relaxation': 'on',
                                  'cleanup_previous_solution': 'on',
                                  'structural_solver': 'NonLinearDynamicPrescribedStep',
                                  'structural_solver_settings':  {'print_info': 'off',
                                                                'max_iterations': 950,
                                                                'delta_curved': 1e-1,
                                                                'min_delta': 1e-6,
                                                                'newmark_damp': 0.5e-4,
                                                                'gravity': True,
                                                                'gravity': 9.81,
                                                                'num_steps': ws.num_steps,
                                                                'dt':ws.dt,
                                                                },
                                  'aero_solver': 'StepUvlm',
                                  'aero_solver_settings':  {'print_info': 'on',
                                                            'num_cores': 4,
                                                            'convection_scheme': 2,
                                                            'velocity_field_generator': 'SteadyVelocityField',
                                                            'velocity_field_input': {'u_inf': ws.u_inf,
                                                                                    'u_inf_direction': [1., 0., 0.]},
                                                            'rho': ws.rho,
                                                            'n_time_steps': ws.num_steps,
                                                            'vortex_radius': 1e-9,
                                                            'dt': ws.dt,
                                                            'gamma_dot_filtering': 3},
                                #'controller_id': {'controller_tip': 'ControlSurfacePidController'},
                                #'controller_settings': {'controller_tip': {'P': ws.gains[0],
                                                                            #'I': ws.gains[1],
                                                                            #'D': ws.gains[2],
                                                                            #'dt': ws.dt,
                                                                            #'input_type': 'roll',
                                                                            #'controlled_surfaces': 0,
                                                                            #'time_history_input_file': 'roll.csv'}},
                                  'fsi_substeps': 200,
                                  'fsi_tolerance': 1e-6,
                                  'relaxation_factor': 0.1,
                                  'minimum_steps': 1,
                                  'relaxation_steps': 150,
                                  'final_relaxation_factor': 0.0,
                                  'n_time_steps': ws.num_steps,
                                  'dt': ws.dt,
                                  'include_unsteady_force_contribution': True,
                                  'postprocessors': ['WriteVariablesTime', 'BeamPlot', 'AerogridPlot'],
                                  'postprocessors_settings': {'BeamPlot': {'include_rbm': 'on',
                                                                           'include_applied_forces': 'on'},
                                                              'StallCheck': {},
                                                              'AerogridPlot': {
                                                                  'u_inf': ws.u_inf,
                                                                  'include_rbm': 'on',
                                                                  'include_applied_forces': 'on',
                                                                  'minus_m_star': 0},
                                                              'WriteVariablesTime': {
                                                                  'structure_variables': ['pos', 'psi'],
                                                                  'structure_nodes': [ws.num_node_surf - 1,
                                                                                      ws.num_node_surf,
                                                                                      ws.num_node_surf + 1],
                                                                    },
                                                                    },
                                    'network_settings': {},
                                    }



ws.config.write()

: 

In [ ]:
sharpy_output = sharpy.sharpy_main.main(['', ws.route + ws.case_name + '.sharpy'])

--------------------------------------------------------------------------------
            ######  ##     ##    ###    ########  ########  ##    ##
           ##    ## ##     ##   ## ##   ##     ## ##     ##  ##  ##
           ##       ##     ##  ##   ##  ##     ## ##     ##   ####
            ######  ######### ##     ## ########  ########     ##
                 ## ##     ## ######### ##   ##   ##           ##
           ##    ## ##     ## ##     ## ##    ##  ##           ##
            ######  ##     ## ##     ## ##     ## ##           ##
--------------------------------------------------------------------------------
Aeroelastics Lab, Aeronautics Department.
    Copyright (c), Imperial College London.
    All rights reserved.
    License available at https://github.com/imperialcollegelondon/sharpy
Running SHARPy from /home/password/sharpy/sharpy
SHARPy being run is in /home/password/.local/lib/python3.10/site-packages
SHARPy output folder set
	/home/password/sharpy/sharpy/output//

fatal: not a git repository (or any of the parent directories): .git


|  1  |  1  |  -2.31093  | -0.4429  |  0.0003  | 12.3470  | -0.0001  |  2.7478  | -0.0000  |
|  2  |  1  |  -4.85716  | -0.4380  |  0.0011  | 12.4428  | -0.0007  |  2.7521  | -0.0001  |
|  3  |  1  |  -4.22749  | -0.4323  |  0.0011  | 12.4119  | -0.0007  |  2.7534  | -0.0001  |
|  4  |  1  |  -4.79342  | -0.4339  |  0.0010  | 12.4216  | -0.0007  |  2.7531  | -0.0001  |
|  5  |  1  |  -5.73425  | -0.4337  |  0.0010  | 12.4204  | -0.0007  |  2.7531  | -0.0001  |
|  0  |  2  |  0.00000   |  0.0249  |  0.0027  | 15.5540  | -0.0010  |  4.1614  | -0.0002  |
|  1  |  2  |  -1.84154  | -1.0248  |  0.0075  | 16.3728  | -0.0027  |  3.7634  | -0.0006  |
|  2  |  2  |  -1.95755  | -0.2791  |  0.0093  | 16.7638  | -0.0039  |  4.0913  | -0.0007  |
|  3  |  2  |  -2.08444  | -0.8354  |  0.0063  | 16.6815  | -0.0023  |  3.8574  | -0.0005  |
|  4  |  2  |  -2.20592  | -0.4105  |  0.0102  | 16.8394  | -0.0042  |  4.0411  | -0.0007  |
|  5  |  2  |  -2.33106  | -0.7266  |  0.0061  | 16.7707  | -0.0023  |

In [ ]:
def get_resulting_vertical_tip_displacement(output_folder, model):
    file_results = os.path.join(output_folder,
                                 model.case_name,
                                 'WriteVariablesTime',
                                 'struct_pos_node{}.dat'.format(model.num_node_surf))
    vertical_tip_displacement = np.loadtxt(file_results)[:,-1]


    return vertical_tip_displacement


tip_displacement_open_loop = get_resulting_vertical_tip_displacement("./output/%s/" %case_name,
                                                                     ws)


time_array = np.arange(0, len(tip_displacement_open_loop) * ws.dt, ws.dt)
normalised_tip_displacement = tip_displacement_open_loop/ (0.5*ws.b_ref) #normalise by half wing span
normalised_tip_displacement *=100# cconvert to percentnvert to percent
plt.plot(time_array, normalised_tip_displacement)
plt.xlabel('time, sec')
plt.ylabel('$z_{tip}/(b_{ref}/2)$, %')
plt.xlim([0., 4])
plt.grid()
plt.show()